# ResNet-18 이미지 분류 예제

Kaggle Notebook 환경에서 Hugging Face `transformers`와 `datasets`를 사용해 `microsoft/resnet-18` 모델로 이미지 분류 추론을 수행합니다.

이 노트북은 다음 흐름으로 구성됩니다.

1. 필요한 라이브러리 설치 및 불러오기
2. `datasets` 라이브러리로 샘플 이미지 로드
3. `AutoImageProcessor`로 이미지 전처리
4. PyTorch 기반 모델 추론
5. 가장 높은 확률의 클래스와 top-10 결과 출력

In [ ]:
# Kaggle Notebook에서 datasets/transformers가 없는 경우를 대비해 설치합니다.
# 이미 설치되어 있다면 대부분 빠르게 넘어갑니다.
%pip install -q transformers datasets pillow

In [ ]:
import torch
from datasets import load_dataset
from IPython.display import display
from transformers import AutoImageProcessor, AutoModelForImageClassification

In [ ]:
# GPU가 있으면 GPU를 사용하고, 없으면 CPU를 사용합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

## 1. datasets 라이브러리에서 샘플 이미지 불러오기

In [ ]:
# Hugging Face datasets Hub에 있는 작은 샘플 이미지 데이터셋을 불러옵니다.
# Kaggle에서 이 셀을 실행하려면 Internet 옵션이 켜져 있어야 합니다.
dataset = load_dataset("huggingface/cats-image")

# 데이터셋의 첫 번째 이미지를 사용합니다.
image = dataset["test"][0]["image"]

# PIL 이미지 형식으로 확인합니다.
print(type(image))
print(f"이미지 크기: {image.size}")
display(image)

## 2. microsoft/resnet-18 모델과 AutoImageProcessor 불러오기

In [ ]:
model_name = "microsoft/resnet-18"

# AutoImageProcessor는 모델이 학습될 때 사용한 resize, crop, normalize 설정을 자동으로 가져옵니다.
image_processor = AutoImageProcessor.from_pretrained(model_name)

# 이미지 분류용 ResNet-18 모델을 불러옵니다.
model = AutoModelForImageClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"모델 로드 완료: {model_name}")

## 3. 이미지 전처리

In [ ]:
# AutoImageProcessor가 PIL 이미지를 PyTorch 텐서 형태로 변환합니다.
inputs = image_processor(images=image, return_tensors="pt")

# 모델과 같은 장치로 입력 텐서를 이동합니다.
inputs = {key: value.to(device) for key, value in inputs.items()}

for key, value in inputs.items():
    print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}, device={value.device}")

## 4. PyTorch 기반 추론 수행

In [ ]:
# torch.no_grad()는 추론 시 gradient 계산을 비활성화해 메모리 사용량을 줄입니다.
with torch.no_grad():
    outputs = model(**inputs)

# logits는 각 클래스에 대한 모델의 원시 점수입니다.
logits = outputs.logits

# softmax를 적용해 클래스별 확률로 변환합니다.
probabilities = torch.softmax(logits, dim=-1)[0]

print(f"logits shape: {tuple(logits.shape)}")
print(f"클래스 개수: {probabilities.shape[0]}")

## 5. 가장 높은 확률의 클래스 출력

In [ ]:
# 가장 높은 확률을 가진 클래스 ID를 찾습니다.
predicted_class_id = probabilities.argmax().item()
predicted_label = model.config.id2label[predicted_class_id]
predicted_probability = probabilities[predicted_class_id].item()

print("가장 높은 확률의 클래스")
print(f"class id   : {predicted_class_id}")
print(f"label      : {predicted_label}")
print(f"probability: {predicted_probability:.4%}")

## 6. Top-10 결과 출력

In [ ]:
# 확률이 높은 순서대로 top-10 클래스를 가져옵니다.
top_k = 10
top_probabilities, top_class_ids = torch.topk(probabilities, k=top_k)

print("Top-10 예측 결과")
print("-" * 70)
print(f"{'rank':>4} | {'class_id':>8} | {'probability':>12} | label")
print("-" * 70)

for rank, (class_id_tensor, probability_tensor) in enumerate(zip(top_class_ids, top_probabilities), start=1):
    class_id = class_id_tensor.item()
    label = model.config.id2label[class_id]
    probability = probability_tensor.item()
    print(f"{rank:>4} | {class_id:>8} | {probability:>11.4%} | {label}")